In [3]:
import argparse
import os
import torch

from exp.exp_informer import Exp_Informer

In [6]:
parser = argparse.ArgumentParser(description='[Informer] Long Sequences Forecasting')
#选择模型
parser.add_argument('--model', type=str, required=True, default='informer',help='model of experiment, options: [informer, informerstack, informerlight(TBD)]')
# 数据集名
parser.add_argument('--data', type=str, required=True, default='WTH', help='data')
# 数据文件根路径
parser.add_argument('--root_path', type=str, default='./data/WTH/', help='root path of the data file')
# 数据文件名
parser.add_argument('--data_path', type=str, default='WTH.csv', help='data file')    
# M ： 多元预测多元， S ： 单变量预测单变量， MS ： 多元预测单变量
parser.add_argument('--features', type=str, default='M', help='forecasting task, options:[M, S, MS]; M:multivariate predict multivariate, S:univariate predict univariate, MS:multivariate predict univariate')
# 标签名
parser.add_argument('--target', type=str, default='OT', help='target feature in S or MS task')
# 重新调整数据间隔（数据重采样）
parser.add_argument('--freq', type=str, default='h', help='freq for time features encoding, options:[s:secondly, t:minutely, h:hourly, d:daily, b:business days, w:weekly, m:monthly], you can also use more detailed freq like 15min or 3h')
# 模型保存的位置
parser.add_argument('--checkpoints', type=str, default='./checkpoints/', help='location of model checkpoints')
# 编码器输入序列长度
parser.add_argument('--seq_len', type=int, default=96, help='input sequence length of Informer encoder')
# 解码器输入序列长度
parser.add_argument('--label_len', type=int, default=48, help='start token length of Informer decoder')
# 预测序列长度
parser.add_argument('--pred_len', type=int, default=24, help='prediction sequence length')
# Informer decoder input: concat[start token series(label_len), zero padding series(pred_len)]
# 编码器输入维度
parser.add_argument('--enc_in', type=int, default=7, help='encoder input size')
# 解码器输入维度
parser.add_argument('--dec_in', type=int, default=7, help='decoder input size')
# 预测未来多少个值？
parser.add_argument('--c_out', type=int, default=7, help='output size')
# 隐藏特征
parser.add_argument('--d_model', type=int, default=512, help='dimension of model')
# 多层注意机制
parser.add_argument('--n_heads', type=int, default=8, help='num of heads')
# 编码器层数
parser.add_argument('--e_layers', type=int, default=2, help='num of encoder layers')
# 解码器层数
parser.add_argument('--d_layers', type=int, default=1, help='num of decoder layers')
# 堆叠的次数
parser.add_argument('--s_layers', type=str, default='3,2,1', help='num of stack encoder layers')

parser.add_argument('--d_ff', type=int, default=2048, help='dimension of fcn')
# 采样因子数，可调整
parser.add_argument('--factor', type=int, default=5, help='probsparse attn factor')

parser.add_argument('--padding', type=int, default=0, help='padding type')

parser.add_argument('--distil', action='store_false', help='whether to use distilling in encoder, using this argument means not using distilling', default=True)
# 丢包
parser.add_argument('--dropout', type=float, default=0.05, help='dropout')
# attention?
parser.add_argument('--attn', type=str, default='prob', help='attention used in encoder, options:[prob, full]')

parser.add_argument('--embed', type=str, default='timeF', help='time features encoding, options:[timeF, fixed, learned]')
# 激活函数
parser.add_argument('--activation', type=str, default='gelu',help='activation')
# 是否输出attention
parser.add_argument('--output_attention', action='store_true', help='whether to output attention in ecoder')
# 是否预测未来数据
parser.add_argument('--do_predict', action='store_true', help='whether to predict unseen future data')
# 是否在生成解码器中使用混合注意力，使用此参数意味着不使用混合注意力
parser.add_argument('--mix', action='store_false', help='use mix attention in generative decoder', default=True)
# 数据文件中的指定列作为输入要素
parser.add_argument('--cols', type=str, nargs='+', help='certain cols from the data files as the input features')
# windows默认为0，不做调整
parser.add_argument('--num_workers', type=int, default=0, help='data loader num workers')
# 训练的轮数
parser.add_argument('--itr', type=int, default=2, help='experiments times')
# 训练迭代次数
parser.add_argument('--train_epochs', type=int, default=6, help='train epochs')
# 批次大小
parser.add_argument('--batch_size', type=int, default=32, help='batch size of train input data')
# 早停
parser.add_argument('--patience', type=int, default=3, help='early stopping patience')
# 学习率
parser.add_argument('--learning_rate', type=float, default=0.0001, help='optimizer learning rate')
#不做调整
parser.add_argument('--des', type=str, default='test',help='exp description')
# 损失函数
parser.add_argument('--loss', type=str, default='mse',help='loss function')
# 不做调整
parser.add_argument('--lradj', type=str, default='type1',help='adjust learning rate')
# 不做调整
parser.add_argument('--use_amp', action='store_true', help='use automatic mixed precision training', default=False)
# 不做调整
parser.add_argument('--inverse', action='store_true', help='inverse output data', default=False)
# 是否使用GPU
parser.add_argument('--use_gpu', type=bool, default=False, help='use gpu')
# 使用GPU的ID号
parser.add_argument('--gpu', type=int, default=0, help='gpu')
# 是否使用多个GPU
parser.add_argument('--use_multi_gpu', action='store_true', help='use multiple gpus', default=False)
# 多路GPU的设备ID
parser.add_argument('--devices', type=str, default='0,1,2,3',help='device ids of multile gpus')

args = parser.parse_args()

args.use_gpu = True if torch.cuda.is_available() and args.use_gpu else False

if args.use_gpu and args.use_multi_gpu:
    args.devices = args.devices.replace(' ','')
    device_ids = args.devices.split(',')
    args.device_ids = [int(id_) for id_ in device_ids]
    args.gpu = args.device_ids[0]

data_parser = {
    'ETTh1':{'data':'ETTh1.csv','T':'OT','M':[7,7,7],'S':[1,1,1],'MS':[7,7,1]},
    'ETTh2':{'data':'ETTh2.csv','T':'OT','M':[7,7,7],'S':[1,1,1],'MS':[7,7,1]},
    'ETTm1':{'data':'ETTm1.csv','T':'OT','M':[7,7,7],'S':[1,1,1],'MS':[7,7,1]},
    'ETTm2':{'data':'ETTm2.csv','T':'OT','M':[7,7,7],'S':[1,1,1],'MS':[7,7,1]},
    'WTH':{'data':'WTH.csv','T':'WetBulbCelsius','M':[12,12,12],'S':[1,1,1],'MS':[12,12,1]},
    'ECL':{'data':'ECL.csv','T':'MT_320','M':[321,321,321],'S':[1,1,1],'MS':[321,321,1]},
    'Solar':{'data':'solar_AL.csv','T':'POWER_136','M':[137,137,137],'S':[1,1,1],'MS':[137,137,1]},
}
if args.data in data_parser.keys():
    data_info = data_parser[args.data]
    args.data_path = data_info['data']
    args.target = data_info['T']
    args.enc_in, args.dec_in, args.c_out = data_info[args.features]

args.s_layers = [int(s_l) for s_l in args.s_layers.replace(' ','').split(',')]
args.detail_freq = args.freq
args.freq = args.freq[-1:]

print('Args in experiment:')
print(args)

Exp = Exp_Informer

for ii in range(args.itr):
    # setting record of experiments
    setting = '{}_{}_ft{}_sl{}_ll{}_pl{}_dm{}_nh{}_el{}_dl{}_df{}_at{}_fc{}_eb{}_dt{}_mx{}_{}_{}'.format(args.model, args.data, args.features, 
                args.seq_len, args.label_len, args.pred_len,
                args.d_model, args.n_heads, args.e_layers, args.d_layers, args.d_ff, args.attn, args.factor, 
                args.embed, args.distil, args.mix, args.des, ii)

    exp = Exp(args) # set experiments
    print('>>>>>>>start training : {}>>>>>>>>>>>>>>>>>>>>>>>>>>'.format(setting))
    exp.train(setting)
    
    print('>>>>>>>testing : {}<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<'.format(setting))
    exp.test(setting)

    if args.do_predict:
        print('>>>>>>>predicting : {}<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<'.format(setting))
        exp.predict(setting, True)

    torch.cuda.empty_cache()

usage: ipykernel_launcher.py [-h] --model MODEL --data DATA [--root_path ROOT_PATH] [--data_path DATA_PATH]
                             [--features FEATURES] [--target TARGET] [--freq FREQ] [--checkpoints CHECKPOINTS]
                             [--seq_len SEQ_LEN] [--label_len LABEL_LEN] [--pred_len PRED_LEN] [--enc_in ENC_IN]
                             [--dec_in DEC_IN] [--c_out C_OUT] [--d_model D_MODEL] [--n_heads N_HEADS]
                             [--e_layers E_LAYERS] [--d_layers D_LAYERS] [--s_layers S_LAYERS] [--d_ff D_FF]
                             [--factor FACTOR] [--padding PADDING] [--distil] [--dropout DROPOUT] [--attn ATTN]
                             [--embed EMBED] [--activation ACTIVATION] [--output_attention] [--do_predict] [--mix]
                             [--cols COLS [COLS ...]] [--num_workers NUM_WORKERS] [--itr ITR]
                             [--train_epochs TRAIN_EPOCHS] [--batch_size BATCH_SIZE] [--patience PATIENCE]
                            

SystemExit: 2